In [1]:
import pandas as pd

# Load dataset
train_df = pd.read_csv("/content/phm_train.csv")
test_df = pd.read_csv("/content/phm_test.csv")

print(train_df.head())
print(train_df['label'].value_counts())


       tweet_id  label                                              tweet
0  6.430000e+17      0  user_mention all i can tell you is i have had ...
1  6.440000e+17      0  my doctor told me stop he gave me sum pop i mi...
2  8.150000e+17      1  i take tylenol and i wake up in the middle of ...
3  6.820000e+17      0  i got xans in an advil bottle i dont take them...
4  6.440000e+17      1  mom says i need to stop eating so much bc ive ...
label
0    7091
1    2900
Name: count, dtype: int64


In [2]:
print(train_df['label'].value_counts(normalize=True))


label
0    0.709739
1    0.290261
Name: proportion, dtype: float64


In [3]:
import re

def clean_tweet(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags symbol only
    text = re.sub(r'#', '', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

train_df['clean_text'] = train_df['tweet'].apply(clean_tweet)
test_df['clean_text'] = test_df['tweet'].apply(clean_tweet)


In [4]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

nltk.download('punkt_tab')


train_df['tokens'] = train_df['clean_text'].apply(word_tokenize)
test_df['tokens'] = test_df['clean_text'].apply(word_tokenize)


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [5]:
from collections import Counter

# Build vocabulary
counter = Counter()

for tokens in train_df['tokens']:
    counter.update(tokens)

# Minimum frequency threshold
min_freq = 2

vocab = {word for word, freq in counter.items() if freq >= min_freq}

# Add special tokens
word2idx = {'<PAD>': 0, '<UNK>': 1}

for word in vocab:
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

vocab_size = len(word2idx)
print("Vocabulary size:", vocab_size)


Vocabulary size: 5442


In [6]:
def encode(tokens, word2idx):
    return [word2idx.get(word, word2idx['<UNK>']) for word in tokens]

train_df['encoded'] = train_df['tokens'].apply(lambda x: encode(x, word2idx))
test_df['encoded'] = test_df['tokens'].apply(lambda x: encode(x, word2idx))


In [7]:
from torch.nn.utils.rnn import pad_sequence
import torch

MAX_LEN = 50

def pad_sequence_custom(seq, max_len):
    if len(seq) > max_len:
        return seq[:max_len]
    else:
        return seq + [0] * (max_len - len(seq))

train_padded = [pad_sequence_custom(seq, MAX_LEN) for seq in train_df['encoded']]
test_padded = [pad_sequence_custom(seq, MAX_LEN) for seq in test_df['encoded']]

X_train = torch.tensor(train_padded)
X_test = torch.tensor(test_padded)

y_train = torch.tensor(train_df['label'].values, dtype=torch.float32)
y_test = torch.tensor(test_df['label'].values, dtype=torch.float32)


In [8]:
!wget http://nlp.stanford.edu/data/glove.twitter.27B.zip
!unzip glove.twitter.27B.zip


--2026-02-19 11:54:42--  http://nlp.stanford.edu/data/glove.twitter.27B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.twitter.27B.zip [following]
--2026-02-19 11:54:42--  https://nlp.stanford.edu/data/glove.twitter.27B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip [following]
--2026-02-19 11:54:43--  https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520408563 (1.4G) [ap

In [9]:
import numpy as np

embedding_dim = 200

# Load GloVe
glove_path = "glove.twitter.27B.200d.txt"

embeddings_index = {}

with open(glove_path, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

print("Loaded GloVe vectors:", len(embeddings_index))


Loaded GloVe vectors: 1193514


In [10]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in word2idx.items():
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)


In [11]:
from torch.utils.data import Dataset, DataLoader

class TweetDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32

train_dataset = TweetDataset(X_train, y_train)
test_dataset = TweetDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)


Bi-LSTM **Model**

In [12]:
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

# Split training into train + validation
X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.15,
    stratify=y_train,
    random_state=42
)

# Convert to PyTorch tensors
train_dataset = TensorDataset(
    torch.tensor(X_train_final, dtype=torch.long),
    torch.tensor(y_train_final, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(X_val, dtype=torch.long),
    torch.tensor(y_val, dtype=torch.float32)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.long),
    torch.tensor(y_test, dtype=torch.float32)
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)


/tmp/ipython-input-2634185891.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_train_final, dtype=torch.long),
/tmp/ipython-input-2634185891.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(y_train_final, dtype=torch.float32)
/tmp/ipython-input-2634185891.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  torch.tensor(X_val, dtype=torch.long),
/tmp/ipython-input-2634185891.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach

In [13]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [15]:
import torch
import torch.nn as nn

class BiLSTM_Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_matrix):
        super(BiLSTM_Model, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=True
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)

        _, (hidden, _) = self.lstm(embedded)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        combined = torch.cat((forward_hidden, backward_hidden), dim=1)

        out = self.dropout(combined)
        out = self.fc(out)

        return out


In [16]:
import torch
import torch.nn as nn

class BiLSTM_Model(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_matrix):
        super(BiLSTM_Model, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=True
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(0.4)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        embedded = self.embedding(x)

        _, (hidden, _) = self.lstm(embedded)

        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        combined = torch.cat((forward_hidden, backward_hidden), dim=1)

        out = self.dropout(combined)
        out = self.fc(out)

        return out


In [17]:
hidden_dim = 256

model = BiLSTM_Model(vocab_size, embedding_dim, hidden_dim, embedding_matrix)
model = model.to(device)


In [18]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

y_train_np = np.array(y_train_final)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0, 1]),
    y=y_train_np
)

pos_weight = torch.tensor([class_weights[1]], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)


In [19]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.0003)


In [20]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

def evaluate(model, loader, threshold=0.5):

    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)
            probs = torch.sigmoid(outputs)

            preds = (probs > threshold).float()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    all_preds = np.array(all_preds).flatten()
    all_labels = np.array(all_labels).flatten()

    acc = accuracy_score(all_labels, all_preds)

    print("Validation Accuracy:", round(acc * 100, 2), "%")
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, digits=4))

    return acc


In [21]:
def train_model(model, train_loader, val_loader, epochs=20):

    best_acc = 0
    patience = 3
    counter = 0

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"\nEpoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

        # 🔹 Validate
        val_acc = evaluate(model, val_loader)

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_bilstm.pt")  # Important: bilstm name
            counter = 0
        else:
            counter += 1

        if counter >= patience:
            print("\nEarly stopping triggered.")
            break

    print("\nBest Validation Accuracy:", round(best_acc * 100, 2), "%")


In [22]:
train_model(model, train_loader, val_loader, epochs=20)



Epoch 1, Loss: 0.6446075374470618
Validation Accuracy: 77.32 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.9200    0.7453    0.8235      1064
         1.0     0.5746    0.8414    0.6828       435

    accuracy                         0.7732      1499
   macro avg     0.7473    0.7933    0.7532      1499
weighted avg     0.8197    0.7732    0.7827      1499


Epoch 2, Loss: 0.5245664160614624
Validation Accuracy: 82.39 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.9040    0.8412    0.8715      1064
         1.0     0.6680    0.7816    0.7203       435

    accuracy                         0.8239      1499
   macro avg     0.7860    0.8114    0.7959      1499
weighted avg     0.8355    0.8239    0.8276      1499


Epoch 3, Loss: 0.5011487460898277
Validation Accuracy: 83.99 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.9023 